In [0]:
%pip install loguru
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 02 — Silver Transformations: Data Quality & Cleansing

Reads the Bronze Delta table, applies data quality checks (drift detection,
frequency compliance, operational mode validation), and writes validated
records to the **Silver** Delta table.

| Property | Value |
|----------|-------|
| **Medallion Layer** | Silver (validated, cleansed) |
| **Source** | `bronze_sensor_readings` Delta table |
| **Sink** | `silver_sensor_readings_validated` Delta table |
| **Quality Log** | `quality_metrics` Delta table |
| **QC Strategy** | Flag → log → filter (only clean records to Silver) |
| **Unity Catalog** | `mining_ops.predictive_maintenance.silver_sensor_readings_validated` |

**Mining context:** Sensor drift is a leading cause of false maintenance alerts
in mining operations. This layer catches drifted thermocouples and pressure
transducers before they contaminate downstream ML models.

## 1. Install Dependencies

The `src.data_quality` module must be available on the cluster.

**Option A — Workspace files (recommended):** Add this repo via Databricks Repos.
The `src/` directory is automatically on `sys.path`.

**Option B — `%pip install`:** If using a wheel or PyPI package:
```python
%pip install /Volumes/main/default/packages/predictive_maintenance-0.1.0-py3-none-any.whl
```

In [0]:
import sys

repo_root = "/Workspace/Users/catherine.varas.padilla@gmail.com/predictive-maintenance-lakehouse"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
    print(f"Added to sys.path: {repo_root}")

## 2. Configuration

In [0]:
dbutils.widgets.text("bronze_path", "/Volumes/workspace/default/bronze/sensor_readings", "Bronze Delta path")
dbutils.widgets.text("silver_path", "/Volumes/workspace/default/silver/sensor_readings_validated", "Silver Delta path")
dbutils.widgets.text("quality_metrics_path", "/Volumes/workspace/default/silver/quality_metrics", "Quality metrics path")
dbutils.widgets.text("drift_window", "20", "Drift detection window size")
dbutils.widgets.text("drift_z_threshold", "3.0", "Drift z-score threshold")

BRONZE_PATH = dbutils.widgets.get("bronze_path")
SILVER_PATH = dbutils.widgets.get("silver_path")
QUALITY_METRICS_PATH = dbutils.widgets.get("quality_metrics_path")
DRIFT_WINDOW = int(dbutils.widgets.get("drift_window"))
DRIFT_Z_THRESHOLD = float(dbutils.widgets.get("drift_z_threshold"))

print(f"Bronze source:    {BRONZE_PATH}")
print(f"Silver sink:      {SILVER_PATH}")
print(f"Quality metrics:  {QUALITY_METRICS_PATH}")
print(f"Drift window:     {DRIFT_WINDOW}")
print(f"Drift z-threshold:{DRIFT_Z_THRESHOLD}")

Bronze source:    /Volumes/workspace/default/bronze/sensor_readings
Silver sink:      /Volumes/workspace/default/silver/sensor_readings_validated
Quality metrics:  /Volumes/workspace/default/silver/quality_metrics
Drift window:     20
Drift z-threshold:3.0


## 3. Imports

In [0]:
from datetime import datetime

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

from src.data_quality import (
    sensor_drift_detection,
    reading_frequency_compliance,
    operational_mode_validation,
    run_all_checks,
)

spark = SparkSession.builder.getOrCreate()

## 4. Read Bronze

In [0]:
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
bronze_count = bronze_df.count()
print(f"Bronze records loaded: {bronze_count:,}")
print(f"Unique units: {bronze_df.select('unit_id').distinct().count()}")

if bronze_count == 0:
    dbutils.notebook.exit("ERROR: Bronze table is empty — nothing to process.")

Bronze records loaded: 900
Unique units: 5


## 5. Apply Data Quality Checks

Convert to Pandas for the `src.data_quality` module, apply all checks,
then convert back to Spark. The module adds `drift_flag` and `op_mode_flag`
columns to identify problematic records.

In [0]:
bronze_pdf = bronze_df.toPandas()

sensor_cols = [c for c in bronze_pdf.columns if c.startswith("s") and c[1:].isdigit()]
op_setting_cols = [c for c in bronze_pdf.columns if c.startswith("op_setting_")]

print(f"Sensor columns ({len(sensor_cols)}): {sensor_cols[:5]}...")
print(f"Op setting columns: {op_setting_cols}")

Sensor columns (21): ['s1', 's2', 's3', 's4', 's5']...
Op setting columns: ['op_setting_1', 'op_setting_2', 'op_setting_3']


### 5a. Sensor Drift Detection

In [0]:
drift_pdf, drift_report = sensor_drift_detection(
    bronze_pdf,
    sensor_cols=sensor_cols,
    window=DRIFT_WINDOW,
    z_threshold=DRIFT_Z_THRESHOLD,
)

print(f"\nDrift report:")
print(f"  Total records:  {drift_report['total_records']:,}")
print(f"  Records flagged:{drift_report['records_flagged']:,}")
print(f"  Flag rate:      {drift_report['flag_rate']:.4f}")

for sensor, count in drift_report["per_sensor_drift_count"].items():
    if count > 0:
        print(f"  {sensor}: {count} drift detections")

2026-04-19 08:24:05.610 | WARNING  | src.data_quality:sensor_drift_detection:101 - Drift detected in s2: 3 readings above z=3.0
2026-04-19 08:24:05.621 | WARNING  | src.data_quality:sensor_drift_detection:101 - Drift detected in s3: 1 readings above z=3.0
2026-04-19 08:24:05.627 | WARNING  | src.data_quality:sensor_drift_detection:101 - Drift detected in s4: 6 readings above z=3.0
2026-04-19 08:24:05.638 | WARNING  | src.data_quality:sensor_drift_detection:101 - Drift detected in s6: 10 readings above z=3.0
2026-04-19 08:24:05.644 | WARNING  | src.data_quality:sensor_drift_detection:101 - Drift detected in s7: 4 readings above z=3.0
2026-04-19 08:24:05.649 | WARNING  | src.data_quality:sensor_drift_detection:101 - Drift detected in s8: 3 readings above z=3.0
2026-04-19 08:24:05.655 | WARNING  | src.data_quality:sensor_drift_detection:101 - Drift detected in s9: 9 readings above z=3.0
2026-04-19 08:24:05.665 | WARNING  | src.data_quality:sensor_drift_detection:101 - Drift detected in s1


Drift report:
  Total records:  900
  Records flagged:33
  Flag rate:      0.0367
  s2: 3 drift detections
  s3: 1 drift detections
  s4: 6 drift detections
  s6: 10 drift detections
  s7: 4 drift detections
  s8: 3 drift detections
  s9: 9 drift detections
  s11: 6 drift detections
  s12: 8 drift detections
  s13: 3 drift detections
  s14: 9 drift detections
  s15: 4 drift detections
  s17: 2 drift detections
  s20: 4 drift detections
  s21: 2 drift detections


### 5b. Reading Frequency Compliance

In [0]:
_, freq_report = reading_frequency_compliance(
    drift_pdf,
    expected_interval_sec=1.0,
)

print(f"\nFrequency compliance:")
print(f"  Quality score:       {freq_report['quality_score']:.4f}")
print(f"  Missing cycles:      {freq_report['total_missing_cycles']}")
print(f"  Duplicate readings:  {freq_report['duplicate_count']}")

2026-04-19 08:24:05.937 | INFO     | src.data_quality:reading_frequency_compliance:203 - Frequency compliance: score=1.0000, 0 missing cycles, 0 duplicates across 5 units



Frequency compliance:
  Quality score:       1.0000
  Missing cycles:      0
  Duplicate readings:  0


### 5c. Operational Mode Validation

In [0]:
validated_pdf, op_report = operational_mode_validation(
    drift_pdf,
    op_setting_cols=op_setting_cols,
)

print(f"\nOperational mode validation:")
print(f"  Records flagged: {op_report['records_flagged']:,}")
print(f"  Flag rate:       {op_report['flag_rate']:.4f}")
for setting, count in op_report["per_setting_violation_count"].items():
    if count > 0:
        print(f"  {setting}: {count} violations")

2026-04-19 08:24:06.110 | INFO     | src.data_quality:operational_mode_validation:279 - Op-mode validation: 0/900 flagged (0.00%)



Operational mode validation:
  Records flagged: 0
  Flag rate:       0.0000


## 6. Log Quality Metrics to Delta

Persisting quality metrics enables a separate observability dashboard
to track data quality trends over time.

In [0]:
run_timestamp = datetime.utcnow().isoformat()

metrics_rows = [
    {"check": "sensor_drift", "metric": "records_flagged", "value": float(drift_report["records_flagged"]), "run_timestamp": run_timestamp},
    {"check": "sensor_drift", "metric": "flag_rate", "value": drift_report["flag_rate"], "run_timestamp": run_timestamp},
    {"check": "frequency_compliance", "metric": "quality_score", "value": freq_report["quality_score"], "run_timestamp": run_timestamp},
    {"check": "frequency_compliance", "metric": "missing_cycles", "value": float(freq_report["total_missing_cycles"]), "run_timestamp": run_timestamp},
    {"check": "operational_mode", "metric": "records_flagged", "value": float(op_report["records_flagged"]), "run_timestamp": run_timestamp},
    {"check": "operational_mode", "metric": "flag_rate", "value": op_report["flag_rate"], "run_timestamp": run_timestamp},
]

for sensor, count in drift_report["per_sensor_drift_count"].items():
    metrics_rows.append({
        "check": "sensor_drift",
        "metric": f"drift_count_{sensor}",
        "value": float(count),
        "run_timestamp": run_timestamp,
    })

metrics_sdf = spark.createDataFrame(pd.DataFrame(metrics_rows))
metrics_sdf.write.format("delta").mode("append").save(QUALITY_METRICS_PATH)
print(f"Logged {len(metrics_rows)} quality metrics to {QUALITY_METRICS_PATH}")

Logged 27 quality metrics to /Volumes/workspace/default/silver/quality_metrics


## 7. Filter to Clean Records & Write Silver

Only records passing **all** quality checks are promoted to Silver.
Flagged records remain in Bronze for investigation.

In [0]:
clean_pdf = validated_pdf[
    ~validated_pdf["drift_flag"] & ~validated_pdf["op_mode_flag"]
].copy()
clean_pdf = clean_pdf.drop(columns=["drift_flag", "op_mode_flag"], errors="ignore")

rejected_count = len(validated_pdf) - len(clean_pdf)
print(f"Records passing QC: {len(clean_pdf):,} / {len(validated_pdf):,}")
print(f"Records rejected:   {rejected_count:,} ({rejected_count / max(len(validated_pdf), 1) * 100:.2f}%)")

if len(clean_pdf) == 0:
    dbutils.notebook.exit("ERROR: All records rejected by quality checks — review drift/op thresholds.")

Records passing QC: 867 / 900
Records rejected:   33 (3.67%)


In [0]:
silver_sdf = spark.createDataFrame(clean_pdf)
silver_sdf = silver_sdf.withColumn("_silver_timestamp", current_timestamp())

(
    silver_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .save(SILVER_PATH)
)
print(f"Silver table written to {SILVER_PATH}")

Silver table written to /Volumes/workspace/default/silver/sensor_readings_validated


## 8. Validate

In [0]:
silver_result = spark.read.format("delta").load(SILVER_PATH)
silver_count = silver_result.count()

print("=" * 60)
print("SILVER TRANSFORMATION SUMMARY")
print("=" * 60)
print(f"  Bronze input:         {bronze_count:,} records")
print(f"  Silver output:        {silver_count:,} records")
print(f"  Records rejected:     {bronze_count - silver_count:,}")
print(f"  Rejection rate:       {(bronze_count - silver_count) / max(bronze_count, 1) * 100:.2f}%")
print(f"  Drift detections:     {drift_report['records_flagged']:,}")
print(f"  Frequency score:      {freq_report['quality_score']:.4f}")
print(f"  Op-mode violations:   {op_report['records_flagged']:,}")
print("=" * 60)

silver_result.printSchema()

SILVER TRANSFORMATION SUMMARY
  Bronze input:         900 records
  Silver output:        867 records
  Records rejected:     33
  Rejection rate:       3.67%
  Drift detections:     33
  Frequency score:      1.0000
  Op-mode violations:   0
root
 |-- unit_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- op_setting_1: double (nullable = true)
 |-- op_setting_2: double (nullable = true)
 |-- op_setting_3: double (nullable = true)
 |-- s1: double (nullable = true)
 |-- s2: double (nullable = true)
 |-- s3: double (nullable = true)
 |-- s4: double (nullable = true)
 |-- s5: double (nullable = true)
 |-- s6: double (nullable = true)
 |-- s7: double (nullable = true)
 |-- s8: double (nullable = true)
 |-- s9: double (nullable = true)
 |-- s10: double (nullable = true)
 |-- s11: double (nullable = true)
 |-- s12: double (nullable = true)
 |-- s13: double (nullable = true)
 |-- s14: double (nullable = true)
 |-- s15: double (nullable = true)
 |-- s16: double (nullable

## 9. Sample Clean Data

In [0]:
display(silver_result.orderBy("unit_id", "cycle").limit(20))

unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,timestamp,_ingestion_timestamp,_source_file,_pipeline_version,_ingestion_date,_silver_timestamp
1,1,-7.0E-4,-4.0E-4,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.419,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,2,0.0019,-3.0E-4,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.0,23.4236,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,3,-0.0043,3.0E-4,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,4,7.0E-4,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,5,-0.0019,-2.0E-4,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393.0,2388.0,100.0,38.9,23.4044,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,6,-0.0043,-1.0E-4,100.0,518.67,642.1,1584.47,1398.37,14.62,21.61,554.67,2388.02,9049.68,1.3,47.16,521.68,2388.03,8132.85,8.4108,0.03,391.0,2388.0,100.0,38.98,23.3669,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,7,0.001,1.0E-4,100.0,518.67,642.48,1592.32,1397.77,14.62,21.61,554.34,2388.02,9059.13,1.3,47.36,522.32,2388.03,8132.32,8.3974,0.03,392.0,2388.0,100.0,39.1,23.3774,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,8,-0.0034,3.0E-4,100.0,518.67,642.56,1582.96,1400.97,14.62,21.61,553.85,2388.0,9040.8,1.3,47.24,522.47,2388.03,8131.07,8.4076,0.03,391.0,2388.0,100.0,38.97,23.3106,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,9,8.0E-4,1.0E-4,100.0,518.67,642.12,1590.98,1394.8,14.62,21.61,553.69,2388.05,9046.46,1.3,47.29,521.79,2388.05,8125.69,8.3728,0.03,392.0,2388.0,100.0,39.05,23.4066,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z
1,10,-0.0033,1.0E-4,100.0,518.67,641.71,1591.24,1400.46,14.62,21.61,553.59,2388.05,9051.7,1.3,47.03,521.79,2388.06,8129.38,8.4286,0.03,393.0,2388.0,100.0,38.95,23.4694,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18,2026-04-19T08:24:13.883Z


## 10. Quality Metrics History

In [0]:
quality_history = spark.read.format("delta").load(QUALITY_METRICS_PATH)
display(quality_history.orderBy(col("run_timestamp").desc()).limit(30))

check,metric,value,run_timestamp
sensor_drift,drift_count_s13,3.0,2026-04-19T08:24:06.328820
sensor_drift,drift_count_s17,2.0,2026-04-19T08:24:06.328820
frequency_compliance,missing_cycles,0.0,2026-04-19T08:24:06.328820
sensor_drift,drift_count_s2,3.0,2026-04-19T08:24:06.328820
operational_mode,flag_rate,0.0,2026-04-19T08:24:06.328820
operational_mode,records_flagged,0.0,2026-04-19T08:24:06.328820
sensor_drift,drift_count_s14,9.0,2026-04-19T08:24:06.328820
sensor_drift,drift_count_s1,0.0,2026-04-19T08:24:06.328820
sensor_drift,drift_count_s9,9.0,2026-04-19T08:24:06.328820
sensor_drift,drift_count_s3,1.0,2026-04-19T08:24:06.328820
